In [ ]:
import os
import json
import re
import time
import requests  # Ensure requests is installed: pip install requests
from typing import List, Dict, Any
from dotenv import load_dotenv

# =========================
# Configuration
# =========================
load_dotenv()
API_KEY = os.getenv("OPENROUTER_API_KEY")
if not API_KEY:
    raise RuntimeError("Set OPENROUTER_API_KEY in your environment or .env")

OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"
MODEL_ID = "mistralai/mistral-7b-instruct"

# =========================
# Prompts
# =========================
SYSTEM_PROMPT = "You are a chemical engineering expert. Extract exactly 5 specific keywords and author countries as JSON."
USER_TEMPLATE = """
Extract exactly 5 keywords and all author countries from this:
Title: {title}
Authors: {authors}
Abstract: {abstract}

Return ONLY JSON: {{"keywords": [], "countries": []}}
"""

# =========================
# Robust API Caller
# =========================
def call_openrouter_with_retry(messages: List[Dict[str, str]], max_retries: int = 5):
    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json"
    }
    payload = {
        "model": MODEL_ID,
        "messages": messages,
        "temperature": 0.1,
        "max_tokens": 500
    }

    delay = 2  # Initial delay in seconds
    for i in range(max_retries):
        try:
            response = requests.post(OPENROUTER_URL, headers=headers, json=payload, timeout=60)
            
            # Handle Rate Limiting (429)
            if response.status_code == 429:
                print(f"Rate limit hit (429). Retrying in {delay} seconds...")
                time.sleep(delay)
                delay *= 2  # Exponential backoff
                continue
            
            response.raise_for_status()
            return response.json()["choices"][0]["message"]["content"]

        except requests.exceptions.RequestException as e:
            print(f"Attempt {i+1} failed: {e}")
            if i == max_retries - 1:
                return "{}" # Return empty JSON string on final failure
            time.sleep(delay)
            delay *= 2
    return "{}"

# =========================
# Helpers
# =========================
def parse_json_loose(text: str) -> Dict[str, Any]:
    try:
        m = re.search(r"\{.*\}", text, flags=re.DOTALL)
        return json.loads(m.group(0)) if m else {}
    except:
        return {}

def get_authors(item):
    parts = []
    for a in item.get("authors_structured", []):
        parts.append(f"{a.get('name')} ({a.get('affiliation')})")
    return "; ".join(parts) if parts else item.get("presenting_author", "")

# =========================
# Processing Loop
# =========================
def process_file(input_path: str):
    with open(input_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    results = []
    print(f"Starting extraction for {len(data)} items...")

    for i, item in enumerate(data):
        title = item.get("title") or item.get("topic") or "Unknown"
        abstract = item.get("abstract") or ""
        authors = get_authors(item)

        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": USER_TEMPLATE.format(title=title, authors=authors, abstract=abstract)}
        ]

        print(f"[{i+1}/{len(data)}] Processing: {title[:40]}...")
        raw_response = call_openrouter_with_retry(messages)
        parsed = parse_json_loose(raw_response)

        results.append({
            "title": title,
            "keywords": parsed.get("keywords", []),
            "countries": parsed.get("countries", [])
        })
        
        # Small mandatory sleep to be polite to the API
        time.sleep(0.5)

    return results

if __name__ == "__main__":
    INPUT_PATH = "aiche_file/aiche_papers_3298.json"
    OUT_DIR = "openrouter/extracted"
    os.makedirs(OUT_DIR, exist_ok=True)

    final_data = process_file(INPUT_PATH)

    with open(os.path.join(OUT_DIR, "full_results.json"), "w", encoding="utf-8") as f:
        json.dump(final_data, f, indent=2)

    print("\nDone! Check the 'openrouter/extracted' folder.")

Starting extraction for 584 items...
[1/584] Processing: 179a- Modeling of Interfacial Properties...
[2/584] Processing: 179b- Modeling the Viscosity of Imidazol...
[3/584] Processing: 179c- Techniques for Measuring and Model...
[4/584] Processing: 179d- Thermophysical Properties of Binar...
[5/584] Processing: 179f- High Pressure Sample Environment f...
[6/584] Processing: 179g- Speed of Sound Measurements of Mix...
[7/584] Processing: 311a- Representing Stereoisomer Effects ...
[8/584] Processing: 311b- The Open Force Field “Rosemary” Re...
[9/584] Processing: 311c- Validation of New Bayesian-Optimiz...
[10/584] Processing: 311d- Bayesian Optimization of Lennard-J...
[11/584] Processing: 546a- Investigation and Analysis of the ...
[12/584] Processing: Break...
[13/584] Processing: 546c- Fingering Effects in Oil-Water Int...
[14/584] Processing: 546d- Numerical Simulation of Interfacia...
[15/584] Processing: 546e- Droplet Impact-Induced Chemical Re...
[16/584] Processing: 546f- Effec

In [ ]:
# openrouter_aiche_extractor_ollama.py
# Use Ollama instead of OpenRouter; keep prompts unchanged; batch + sleep + retries

import os
import json
import time
import re
import subprocess
from typing import List, Dict, Any
from dotenv import load_dotenv

load_dotenv()

# ---------------- config ----------------
# Ollama model name (local)
OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "llama3:latest")

ABSTRACT_YEAR = 2023

INPUT_FILES = [
    "aiche_papers_3288.json",
    "aiche_papers_3298.json",
    "aiche_papers_3299.json",
    "aiche_papers_3300.json",
    "aiche_papers_3302.json",
    "aiche_papers_3303.json",
    "aiche_papers_3305.json",
    "aiche_papers_3306.json",
    "aiche_papers_3307.json",
    "aiche_papers_3309.json",
    "aiche_papers_3311.json",
    "aiche_papers_3312.json",
    "aiche_papers_3313.json",
    "aiche_papers_3314.json",
    "aiche_papers_3315.json",
    "aiche_papers_3316.json",
    "aiche_papers_3317.json",
    "aiche_papers_3318.json",
    "aiche_papers_3319.json",
    "aiche_papers_3321.json",
    "aiche_papers_3325.json",
    "aiche_papers_3330.json"
]

OUTPUT_FOLDER = "extracted"
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

# --------- YOUR EXACT PROMPTS (unchanged) ----------
SYSTEM_PROMPT = """You are an expert keyword extractor specialized in chemical engineering.
Analyze AIChE abstracts and output JSON with 'keywords' and 'countries'."""

FEW_SHOT = """### Example
Input:
Title/Topic: CO2 Electroreduction to Multicarbon Products on Copper Nanocubes
Authors/Affiliations: M. Garcia (ETH Zürich, Switzerland)
Abstract: ...
Output JSON:
{"keywords":["CO2 electroreduction","copper nanocubes","C2+ products","electrocatalysis","selectivity tuning"],
 "countries":["Switzerland"]}"""

USER_TEMPLATE = """{few_shot}

Extract exactly 5–10 key chemical engineering keywords and all author countries.
Return JSON:
{{"keywords":["k1","k2"], "countries":["c1"]}}

Input:
Title/Topic: {title}
Authors/Affiliations: {authors}
Abstract: {abstract}

Output JSON:
"""
# ----------------------------------------------------

def authors_block(item: Dict[str, Any]) -> str:
    parts = []
    for a in item.get("authors_structured", []) or []:
        nm = (a.get("name") or "").strip()
        af = (a.get("affiliation") or "").strip()
        if nm or af:
            parts.append(f"{nm} ({af})")
    if not parts and item.get("presenting_author"):
        parts.append(str(item["presenting_author"]))
    return "; ".join(parts)

# Try to use ollama python API first
def call_ollama_via_python(messages: List[Dict[str, str]], max_retries: int = 3, timeout: int = 120) -> str:
    try:
        import ollama
    except Exception:
        raise RuntimeError("Ollama python package not available")

    # build messages in the format ollama.chat expects (if using the client)
    # Some ollama versions expect a single 'messages' list similar to other libs.
    for attempt in range(1, max_retries + 1):
        try:
            # ollama.chat returns a dict-like response (API varies by client version)
            resp = ollama.chat(model=OLLAMA_MODEL, messages=messages, timeout=timeout)
            # Try common fields
            if isinstance(resp, dict):
                # try choices -> message -> content
                if "choices" in resp and resp["choices"]:
                    c = resp["choices"][0]
                    if isinstance(c, dict) and "message" in c and "content" in c["message"]:
                        return c["message"]["content"]
                # sometimes resp has 'answer' or 'output'
                if "answer" in resp:
                    return resp["answer"]
                if "output" in resp:
                    return resp["output"]
                # fallback to string conversion
                return str(resp)
            else:
                # if resp is a string or simple object
                return str(resp)
        except Exception as e:
            if attempt == max_retries:
                raise
            wait = 2 * attempt
            print(f"Ollama python API error (attempt {attempt}/{max_retries}): {e}. Retrying in {wait}s...")
            time.sleep(wait)
    raise RuntimeError("Ollama python API failed after retries")

# Fallback to ollama CLI via subprocess
def call_ollama_via_cli(messages: List[Dict[str, str]], max_retries: int = 3, timeout: int = 120) -> str:
    # Join messages into a single prompt string for CLI
    prompt_parts = []
    # include system as a single block first
    for m in messages:
        role = m.get("role", "user")
        content = m.get("content", "")
        if role == "system":
            prompt_parts.append(f"SYSTEM:\n{content}\n")
        else:
            # prefix each user message to separate them clearly
            prompt_parts.append(f"USER:\n{content}\n")
    prompt = "\n".join(prompt_parts)

    for attempt in range(1, max_retries + 1):
        try:
            # Use `ollama` CLI: `ollama generate <model> --prompt "<prompt>"`
            # We pass the prompt via stdin to avoid shell quoting issues.
            proc = subprocess.run(
                ["ollama", "generate", OLLAMA_MODEL],
                input=prompt.encode("utf-8"),
                stdout=subprocess.PIPE,
                stderr=subprocess.PIPE,
                timeout=timeout
            )
            if proc.returncode != 0:
                stderr = proc.stderr.decode("utf-8", errors="ignore")
                raise RuntimeError(f"ollama CLI error: {stderr.strip()}")
            out = proc.stdout.decode("utf-8", errors="ignore")
            return out
        except Exception as e:
            if attempt == max_retries:
                raise
            wait = 2 * attempt
            print(f"Ollama CLI error (attempt {attempt}/{max_retries}): {e}. Retrying in {wait}s...")
            time.sleep(wait)
    raise RuntimeError("Ollama CLI failed after retries")

def call_ollama(messages: List[Dict[str, str]], max_tokens: int = 1024) -> str:
    # Try python client, then CLI
    try:
        return call_ollama_via_python(messages)
    except Exception as e:
        # print minimal message and fallback to CLI
        print(f"Ollama python client unavailable/failed: {e}. Falling back to CLI.")
        return call_ollama_via_cli(messages)

def extract_json_objects(text: str) -> List[Dict[str, Any]]:
    objs = []
    # find JSON objects - non-greedy braces capture
    for m in re.finditer(r"\{[\s\S]*?\}", text):
        raw = m.group(0)
        try:
            objs.append(json.loads(raw))
        except:
            # try minor fixes
            s = raw.replace("'", '"')
            try:
                objs.append(json.loads(s))
            except:
                pass
    return objs

def process_file_batch(path: str, batch_size: int = 12) -> List[Dict[str, Any]]:
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    if not isinstance(data, list):
        raise ValueError("Input must be a list")

    results = []
    total = len(data)
    i = 0
    while i < total:
        batch = data[i:i+batch_size]

        # Build many USER messages (YOUR TEMPLATE) in one single request
        all_user_messages = []
        for item in batch:
            title = (item.get("title") or item.get("topic") or "").strip()
            abstract = (item.get("abstract") or "").strip()

            # ---------- robust authors handling ----------
            a_field = item.get("authors")
            if isinstance(a_field, str):
                authors = a_field.strip()
            elif isinstance(a_field, list):
                # join list entries; if element is dict, try to format it; else str()
                authors_parts = []
                for a in a_field:
                    if isinstance(a, str):
                        authors_parts.append(a.strip())
                    elif isinstance(a, dict):
                        # try to extract name and affiliation if present
                        name = (a.get("name") or "").strip()
                        aff  = (a.get("affiliation") or "").strip()
                        if name and aff:
                            authors_parts.append(f"{name} ({aff})")
                        elif name:
                            authors_parts.append(name)
                        else:
                            authors_parts.append(json.dumps(a, ensure_ascii=False))
                    else:
                        authors_parts.append(str(a).strip())
                authors = "; ".join([p for p in authors_parts if p])
            elif isinstance(a_field, dict):
                # single structured author-like dict
                name = (a_field.get("name") or "").strip()
                aff  = (a_field.get("affiliation") or "").strip()
                authors = f"{name} ({aff})" if name or aff else authors_block(item)
            else:
                # fallback to authors_block (structured authors) or empty
                authors = (authors_block(item) or "").strip()
            # ------------------------------------------------

            prompt = USER_TEMPLATE.format(
                few_shot=FEW_SHOT,
                title=title,
                authors=authors,
                abstract=abstract
            ).strip()

            all_user_messages.append({"role": "user", "content": prompt})

        # Prepend system message once
        messages = [{"role": "system", "content": SYSTEM_PROMPT}] + all_user_messages

        raw = call_ollama(messages)
        json_list = extract_json_objects(raw)

        # Align output (if model returned fewer/more)
        if not isinstance(json_list, list):
            json_list = []

        while len(json_list) < len(batch):
            json_list.append({"keywords": [], "countries": []})
        json_list = json_list[:len(batch)]

        for item, parsed in zip(batch, json_list):
            title = (item.get("title") or item.get("topic") or "").strip()
            # reuse robust authors extraction for saved output
            a_field = item.get("authors")
            if isinstance(a_field, str):
                authors_out = a_field.strip()
            elif isinstance(a_field, list):
                authors_out = "; ".join([str(x).strip() for x in a_field if x])
            elif isinstance(a_field, dict):
                name = (a_field.get("name") or "").strip()
                aff  = (a_field.get("affiliation") or "").strip()
                authors_out = f"{name} ({aff})" if name or aff else (authors_block(item) or "").strip()
            else:
                authors_out = (authors_block(item) or "").strip()

            results.append({
                "title": title,
                "authors": authors_out,
                "keywords": parsed.get("keywords", []),
                "countries": parsed.get("countries", []),
                "Year": ABSTRACT_YEAR
            })

        i += batch_size
        time.sleep(1.5)

    return results


# ------------------ RUN -------------------
if __name__ == "__main__":
    combined = []
    total_files = len(INPUT_FILES)

    for idx, input_path in enumerate(INPUT_FILES, start=1):
        if not os.path.isfile(input_path):
            print(f"[{idx}/{total_files}] Missing: {input_path}")
            continue

        print(f"[{idx}/{total_files}] Processing {input_path}")

        try:
            results = process_file_batch(input_path, batch_size=12)
        except Exception as e:
            print(f"Error processing {input_path}: {e}")
            raise

        combined.extend(results)

        base = os.path.splitext(os.path.basename(input_path))[0]
        out_path = os.path.join(OUTPUT_FOLDER, f"{base}_full_results_openrouter.json")
        with open(out_path, "w", encoding="utf-8") as f:
            json.dump(results, f, ensure_ascii=False, indent=2)
        print(f"Saved → {out_path}")

    with open("all_combined_full_results.json", "w", encoding="utf-8") as f:
        json.dump(combined, f, ensure_ascii=False, indent=2)
    print("Saved combined file: all_combined_full_results.json")


[1/22] Processing aiche_papers_3288.json
Ollama python API error (attempt 1/3): Client.chat() got an unexpected keyword argument 'timeout'. Retrying in 2s...
Ollama python API error (attempt 2/3): Client.chat() got an unexpected keyword argument 'timeout'. Retrying in 4s...
Ollama python client unavailable/failed: Client.chat() got an unexpected keyword argument 'timeout'. Falling back to CLI.
Ollama CLI error (attempt 1/3): ollama CLI error: Error: unknown command "generate" for "ollama". Retrying in 2s...
Ollama CLI error (attempt 2/3): ollama CLI error: Error: unknown command "generate" for "ollama". Retrying in 4s...
Error processing aiche_papers_3288.json: ollama CLI error: Error: unknown command "generate" for "ollama"


RuntimeError: ollama CLI error: Error: unknown command "generate" for "ollama"

In [1]:
# openrouter_aiche_extractor.py
# Fixed naming inconsistencies and template formatting

import os, json, requests, re
from time import time
from typing import List, Dict, Any
from dotenv import load_dotenv

load_dotenv()
API_KEY = os.getenv("OPENROUTER_API_KEY")
if not API_KEY:
    raise RuntimeError("Missing API key")

OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"
MODEL_ID = "meta-llama/llama-3.3-70b-instruct:free"

ABSTRACT_YEAR = 2023
INPUT_FILES = [
    "aiche_sample.json",
]

OUTPUT_FOLDER = "extracted"
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

SYSTEM_PROMPT = """
You are an expert keyword extractor specialized in chemical engineering.
Your task is to analyze abstracts from AIChE conferences and extract
exactly 5 or 10 of the most important and specific keywords or phrases related to
chemical engineering from each abstract, along with the unique countries
of all authors.

The keywords should:
1. Reflect the core chemical engineering focus of the abstract.
2. Be highly specific (e.g., 'heterogeneous catalysis' instead of 'catalysis').
3. Avoid generic or vague terms (e.g., 'study', 'analysis', 'process').
4. Be formatted as a JSON list of exactly 5 to 10 unique entries without explanation.
"""

FEW_SHOT = """
### Example
Input:
Title/Topic: CO2 Electroreduction to Multicarbon Products on Copper Nanocubes
Authors/Affiliations: M. Garcia (ETH Zürich, Switzerland)
Abstract: Copper nanocube electrodes selectively reduce CO2 to C2+ products via...
Output JSON:
{"keywords": ["CO2 electroreduction", "copper nanocubes", "C2+ products", "electrocatalysis", "selectivity tuning"],
 "countries": ["Switzerland"]}
"""

USER_TEMPLATE = """
{few_shot}
Extract exactly 5 most important chemical engineering keywords and all author countries from this abstract.
Format as JSON:
{{"keywords": ["keyword1", "keyword2", "keyword3", "keyword4", "keyword5"],
  "countries": ["country1", "country2", ...]}}

Input:
Title/Topic: {title_or_topic}
Authors/Affiliations: {authors_block}
Abstract: {abstract_text}

Output JSON:
"""

def get_authors_string(item):
    parts = []
    for a in item.get("authors_structured", []) or []:
        nm = (a.get("name") or "").strip()
        af = (a.get("affiliation") or "").strip()
        if nm or af: parts.append(f"{nm} ({af})")
    if not parts and item.get("presenting_author"):
        parts.append(str(item["presenting_author"]))
    return "; ".join(parts)

def call_openrouter(messages, max_tokens=256):
    headers = {"Authorization": f"Bearer {API_KEY}", "Content-Type": "application/json"}
    payload = {"model": MODEL_ID, "messages": messages, "temperature": 0.0, "max_tokens": max_tokens}
    r = requests.post(OPENROUTER_URL, headers=headers, json=payload, timeout=90)
    r.raise_for_status()
    return r.json()["choices"][0]["message"]["content"]

def parse_json_loose(text):
    try: return json.loads(text)
    except:
        s = text.strip().strip("`")
        m = re.search(r"\{[\s\S]*\}", s)
        if m: s = m.group(0)
        s = s.replace("’","'").replace("“",'"').replace("”",'"')
        s = s.replace("'",'"')
        s = re.sub(r",\s*}", "}", s)
        s = re.sub(r",\s*]", "]", s)
        try: return json.loads(s)
        except: return {"keywords": [], "countries": [], "_raw": text}

def process_file(path):
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    results = []
    for item in data:
        title = (item.get("title") or item.get("topic") or "").strip()
        abstract = (item.get("abstract") or "").strip()
        authors = get_authors_string(item)

        # Mapping placeholders to match USER_TEMPLATE
        user_prompt = USER_TEMPLATE.format(
            few_shot=FEW_SHOT, 
            title_or_topic=title, 
            authors_block=authors, 
            abstract_text=abstract
        )

        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ]

        raw = call_openrouter(messages)
        parsed = parse_json_loose(raw)

        results.append({
            "title": title,
            "authors": authors,
            "keywords": parsed.get("keywords", []),
            "countries": parsed.get("countries", []),
            "Year": ABSTRACT_YEAR
        })
    return results

if __name__ == "__main__":
    combined = []
    total = len(INPUT_FILES)

    for idx, input_path in enumerate(INPUT_FILES, start=1):
        if not os.path.isfile(input_path):
            print(f"[{idx}/{total}] Missing: {input_path}")
            continue

        print(f"[{idx}/{total}] Extracting: {input_path}")

        all_results = process_file(input_path)
        combined.extend(all_results)
        base = os.path.splitext(os.path.basename(input_path))[0]
        OUT_PATH = os.path.join(OUTPUT_FOLDER, f"{base}_full_results_openrouter.json")

        with open(OUT_PATH, "w", encoding="utf-8") as f:
            json.dump(all_results, f, ensure_ascii=False, indent=2)

        print(f"Saved: {OUT_PATH}")

    COMBINED_OUT = "all_combined_full_results_holla.json"
    with open(COMBINED_OUT, "w", encoding="utf-8") as f:
        json.dump(combined, f, ensure_ascii=False, indent=2)

    print(f"Saved combined: {COMBINED_OUT}")

[1/1] Extracting: aiche_sample.json


HTTPError: 429 Client Error: Too Many Requests for url: https://openrouter.ai/api/v1/chat/completions

In [ ]:
# build_aggregate.py
# Build keyword -> {years, countries, authors} -> counts JSON

import os, json, re
from collections import defaultdict, Counter

INPUT_FILE = "extracted/combined.json"
OUTPUT_FILE = "keyword_aggregate_all_results.json"

def canonical_kw(k: str) -> str:
    return k.strip().lower()

def canonical_country(c: str) -> str:
    return c.strip()

def canonical_author(a: str) -> str:
    return a.strip()

def extract_year(item):
    # try Year field first
    y = item.get("Year") or item.get("year")
    if y:
        try:
            return str(int(y))
        except:
            pass
    # fallback: search title or abstract for a 4-digit year 1900-2099
    text = (item.get("title","") or "") + " " + (item.get("abstract","") or "")
    m = re.search(r"\b(19|20)\d{2}\b", text)
    if m:
        return m.group(0)
    return None

def authors_from_authors_field(s: str):
    # authors field looks like "Name (University); Name2 (Univ2)" or single
    parts = []
    for part in re.split(r";|\n", s or ""):
        p = part.strip()
        if p:
            parts.append(p)
    return parts

def main():
    if not os.path.isfile(INPUT_FILE):
        raise SystemExit(f"Missing input: {INPUT_FILE}")

    with open(INPUT_FILE, "r", encoding="utf-8") as f:
        items = json.load(f)

    agg = {}  # keyword -> {"years":{y:count}, "countries":{c:count}, "authors":{a:count}}

    for item in items:
        year = extract_year(item)
        kws = item.get("keywords") or []
        countries = item.get("countries") or []
        authors_field = item.get("authors") or ""
        authors = authors_from_authors_field(authors_field)

        for rawk in kws:
            if not rawk: continue
            k = canonical_kw(rawk)
            rec = agg.setdefault(k, {"years":{}, "countries":{}, "authors":{}})
            if year:
                rec["years"][year] = rec["years"].get(year, 0) + 1
            for c in countries:
                cc = canonical_country(c)
                if cc:
                    rec["countries"][cc] = rec["countries"].get(cc, 0) + 1
            for a in authors:
                aa = canonical_author(a)
                if aa:
                    rec["authors"][aa] = rec["authors"].get(aa, 0) + 1

    # optional: sort years for readability (not required)
    for k, rec in agg.items():
        rec["years"] = dict(sorted(rec["years"].items(), key=lambda x: int(x[0])))
        # sort countries/authors by count desc (keep as dict)
        rec["countries"] = dict(sorted(rec["countries"].items(), key=lambda x: -x[1]))
        rec["authors"] = dict(sorted(rec["authors"].items(), key=lambda x: -x[1]))

    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        json.dump(agg, f, ensure_ascii=False, indent=2)

    print(f"Wrote aggregate: {OUTPUT_FILE} (keywords: {len(agg)})")

if __name__ == "__main__":
    main()


Wrote aggregate: keyword_aggregate_all_results.json (keywords: 11213)


In [ ]:

import os
import json
import re
from collections import defaultdict, Counter
from typing import List, Any

INPUT_FILE = "extracted/aiche_papers_3288_full_results.json"
OUTPUT_FILE = "keyword_aggregate.json"


def canonical_kw(k: str) -> str:
    """Lowercase and trim keyword (keeps internal spacing)."""
    return re.sub(r"\s+", " ", (k or "").strip()).lower()


def canonical_country(c: str) -> str:
    """Strip and title-case country names for consistency."""
    s = (c or "").strip()
    return s.title() if s else s


def canonical_author(a: str) -> str:
    """
    Strip trailing affiliation in parentheses and collapse whitespace.
    E.g. "John Doe (Univ)" -> "John Doe"
    """
    s = (a or "").strip()
    # remove trailing " (affiliation...)" if present
    s = re.sub(r"\s*\([^)]*\)\s*$", "", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s


def extract_year(item: dict):
    # try Year field first
    y = item.get("Year") or item.get("year")
    if y:
        try:
            return str(int(y))
        except Exception:
            pass
    # fallback: search title or abstract for a 4-digit year 1900-2099
    text = (item.get("title", "") or "") + " " + (item.get("abstract", "") or "")
    m = re.search(r"\b(19|20)\d{2}\b", text)
    if m:
        return m.group(0)
    return None


def authors_from_authors_field(s: Any) -> List[str]:
    """
    Parse authors field into a list of author strings.

    Accepts:
      - string like "Name (Affil); Name2 (Affil2)"
      - list of strings or list of dicts
      - dicts with possible 'authors_structured' / 'name' keys

    Returns a list of author strings (trimmed).
    """
    parts: List[str] = []
    if s is None:
        return parts

    # If list: recurse
    if isinstance(s, list):
        for elt in s:
            parts.extend(authors_from_authors_field(elt))
        return [p for p in (x.strip() for x in parts) if p]

    # If dict: try to extract structured authors or name/affiliation
    if isinstance(s, dict):
        if "authors_structured" in s and isinstance(s["authors_structured"], list):
            for a in s["authors_structured"]:
                if isinstance(a, dict):
                    name = (a.get("name") or "").strip()
                    aff = (a.get("affiliation") or "").strip()
                    combined = f"{name} ({aff})" if aff else name
                    if combined.strip():
                        parts.append(combined.strip())
                else:
                    parts.extend(authors_from_authors_field(a))
            return [p for p in (x.strip() for x in parts) if p]

        # fallback to name / affiliation keys
        name = (s.get("name") or s.get("author") or "").strip()
        aff = (s.get("affiliation") or s.get("affil") or "").strip()
        if name or aff:
            combined = f"{name} ({aff})" if aff else name
            return [combined.strip()] if combined.strip() else []

        # last resort: stringify
        s = json.dumps(s)

    # s should be a string now
    s = str(s).strip()
    if not s:
        return []

    # split on obvious separators ; or newline or |
    if any(sep in s for sep in (";", "\n", "|")):
        raw_parts = re.split(r";|\n|\|", s)
        for part in raw_parts:
            p = part.strip()
            if p:
                parts.append(p)
        return parts

    # otherwise return as single author string
    return [s]


def main():
    if not os.path.isfile(INPUT_FILE):
        raise SystemExit(f"Missing input: {INPUT_FILE}")

    with open(INPUT_FILE, "r", encoding="utf-8") as f:
        items = json.load(f)

    agg = {}  # keyword -> {"years":{y:count}, "countries":{c:count}, "authors":{a:count}}

    for item in items:
        year = extract_year(item)

        # Normalize keywords to list
        kws = item.get("keywords") or []
        if isinstance(kws, str):
            kws = [kws]
        if kws is None:
            kws = []

        # Normalize countries to list
        countries = item.get("countries") or []
        if isinstance(countries, str):
            countries = [countries]
        if countries is None:
            countries = []

        authors_field = item.get("authors") or item.get("author") or item.get("authors_field") or ""
        authors = authors_from_authors_field(authors_field)

        for rawk in kws:
            if not rawk:
                continue

            # # enforce 1-2 word requirement
            # token_count = len(re.findall(r"\S+", str(rawk)))
            # if token_count < 1 or token_count > 2:
            #     continue

            k = canonical_kw(rawk)
            if not k:
                continue

            rec = agg.setdefault(k, {"years": {}, "countries": {}, "authors": {}})
            if year:
                rec["years"][year] = rec["years"].get(year, 0) + 1
            for c in countries:
                cc = canonical_country(c)
                if cc:
                    rec["countries"][cc] = rec["countries"].get(cc, 0) + 1
            for a in authors:
                aa = canonical_author(a)
                if aa:
                    rec["authors"][aa] = rec["authors"].get(aa, 0) + 1

    # Sort years numerically where possible, and sort countries/authors by descending count
    for k, rec in agg.items():
        try:
            rec["years"] = dict(sorted(rec["years"].items(), key=lambda x: int(x[0])))
        except Exception:
            rec["years"] = dict(sorted(rec["years"].items()))
        rec["countries"] = dict(sorted(rec["countries"].items(), key=lambda x: -x[1]))
        rec["authors"] = dict(sorted(rec["authors"].items(), key=lambda x: -x[1]))

    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        json.dump(agg, f, ensure_ascii=False, indent=2)

    print(f"Wrote aggregate: {OUTPUT_FILE} (keywords: {len(agg)})")


if __name__ == "__main__":
    main()


Wrote aggregate: keyword_aggregate.json (keywords: 330)


In [ ]:
# query_faiss.py
# Robust FAISS query tool with RapidFuzz fallback

import os, json, sys
import numpy as np

SIM_THRESHOLD = 0.90  # cosine threshold (0..1)
TOP_K = 5
FAISS_INDEX = "faiss.index"
ID_MAP = "faiss_id2variant.json"
CLUSTERS_FILE = "clusters.json"
EMB_MODEL = "all-MiniLM-L6-v2"

def load_index():
    try:
        import faiss
    except Exception:
        raise SystemExit("faiss not installed. pip install faiss-cpu")
    if not os.path.isfile(FAISS_INDEX) or not os.path.isfile(ID_MAP):
        raise SystemExit("Index or id-map missing; run build_faiss_index.py first")
    index = faiss.read_index(FAISS_INDEX)
    with open(ID_MAP, "r", encoding="utf-8") as f:
        id2v_raw = json.load(f)
    # normalize id2v keys to int -> variant
    id2v = {}
    for k, v in id2v_raw.items():
        try:
            ik = int(k)
        except:
            # if keys already ints (json numeric) they come as ints
            ik = k
        id2v[int(ik)] = v
    return index, id2v

def embed_query(model, text):
    v = model.encode([text], convert_to_numpy=True)
    v = v.astype("float32")
    norm = np.linalg.norm(v, axis=1, keepdims=True)
    norm[norm==0] = 1.0
    v = v / norm
    return v

def rapidfuzz_fallback(query, clusters_file, top_k=5):
    # very fast fallback: token_sort_ratio against cluster variants
    try:
        from rapidfuzz import process, fuzz
    except Exception:
        raise SystemExit("rapidfuzz not installed. pip install rapidfuzz")
    choices = []
    canon_for_variant = {}
    if os.path.isfile(clusters_file):
        with open(clusters_file, "r", encoding="utf-8") as f:
            clusters = json.load(f)
        for c, vars in clusters.items():
            for v in vars:
                choices.append(v)
                canon_for_variant[v] = c
    if not choices:
        print("No cluster variants available for fuzzy fallback.")
        return []
    res = process.extract(query, choices, scorer=fuzz.token_sort_ratio, limit=top_k)
    out = []
    for match, score, _ in res:
        out.append({"variant": match, "score_token_sort": score/100.0, "canonical": canon_for_variant.get(match)})
    return out

def main():
    # try to load FAISS index and embedding model
    use_faiss = os.path.isfile(FAISS_INDEX) and os.path.isfile(ID_MAP)
    model = None
    index = None
    id2v = {}

    if use_faiss:
        try:
            index, id2v = load_index()
        except Exception as e:
            print("FAISS index load failed:", e)
            use_faiss = False

    if use_faiss:
        try:
            from sentence_transformers import SentenceTransformer
            model = SentenceTransformer(EMB_MODEL)
        except Exception as e:
            print("SentenceTransformer load failed (will fallback to rapidfuzz):", e)
            model = None
            use_faiss = False

    # prepare canonical mapping from clusters.json (optional)
    canon_for_variant = {}
    if os.path.isfile(CLUSTERS_FILE):
        with open(CLUSTERS_FILE, "r", encoding="utf-8") as f:
            clusters = json.load(f)
        for c, vars in clusters.items():
            for v in vars:
                canon_for_variant[v] = c

    print("Ready. Using FAISS+embeddings:" , use_faiss)
    print("Threshold (cosine):", SIM_THRESHOLD)
    try:
        while True:
            q = input("\nQuery keyword (ENTER to quit): ").strip()
            if not q:
                break

            if use_faiss and model is not None and index is not None:
                v = embed_query(model, q)
                D, I = index.search(v, TOP_K)
                sims = D[0].tolist()
                ids = I[0].tolist()

                print(f"\nTop-{TOP_K} FAISS results (inner-product ~ cosine):")
                any_above = False
                for idx, sim in zip(ids, sims):
                    if idx < 0:
                        continue
                    variant = id2v.get(int(idx))
                    canonical = canon_for_variant.get(variant)
                    print(f"  id={idx}  sim={sim:.4f}  variant='{variant}'  canonical='{canonical}'")
                    if sim >= SIM_THRESHOLD:
                        any_above = True
                if any_above:
                    print(f"--> There are matches >= {SIM_THRESHOLD}")
                else:
                    print(f"--> No matches >= {SIM_THRESHOLD}; showing top-{TOP_K} for inspection.")
            else:
                # fallback to rapidfuzz token_sort
                print("\nFAISS/embeddings unavailable — using RapidFuzz fallback (token_sort_ratio).")
                hits = rapidfuzz_fallback(q, CLUSTERS_FILE, top_k=TOP_K)
                if not hits:
                    print("No fuzzy hits.")
                else:
                    for h in hits:
                        print(f"  score={h['score_token_sort']:.3f}  variant='{h['variant']}'  canonical='{h.get('canonical')}'")
                    best = hits[0]
                    if best['score_token_sort'] >= SIM_THRESHOLD:
                        print(f"--> Best fuzzy hit >= {SIM_THRESHOLD}: {best['variant']}")
                    else:
                        print(f"--> Best fuzzy score {best['score_token_sort']:.3f} (< {SIM_THRESHOLD})")
    except KeyboardInterrupt:
        print("\nbye")
    except Exception as e:
        print("Unhandled error:", e)
        raise

if __name__ == "__main__":
    main()


Ready. Using FAISS+embeddings: False
Threshold (cosine): 0.9

FAISS/embeddings unavailable — using RapidFuzz fallback (token_sort_ratio).
No cluster variants available for fuzzy fallback.
No fuzzy hits.

FAISS/embeddings unavailable — using RapidFuzz fallback (token_sort_ratio).
No cluster variants available for fuzzy fallback.
No fuzzy hits.


In [ ]:
import os
import json
import re
import time
import requests
from typing import List, Dict, Any
from dotenv import load_dotenv

# ---------------------------------------------------------
# 1. Configuration
# ---------------------------------------------------------
load_dotenv()
API_KEY = os.getenv("OPENROUTER_API_KEY")
if not API_KEY:
    raise RuntimeError("OPENROUTER_API_KEY not found. Check your .env file.")

OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"
# Using the 72B Model as requested
MODEL_ID = "meta-llama/llama-3.1-405b-instruct:free"

ABSTRACT_YEAR = 2023
INPUT_FOLDER = "dummy"
OUTPUT_FOLDER = "openrouter_extracted"

# ---------------------------------------------------------
# 2. Prompts (Strictly Unchanged)
# ---------------------------------------------------------
System_Prompt = """
You are an expert keyword extractor specialized in chemical engineering.
Your task is to analyze abstracts from AIChE conferences and extract
exactly 5 or 10 of the most important and specific keywords or phrases related to
chemical engineering from each abstract, along with the unique countries
of all authors.

The keywords should:
1. Reflect the core chemical engineering focus of the abstract.
2. Each keyword should be either one word or two words—no longer phrases allowed.
3. Be highly specific (e.g., 'heterogeneous catalysis' instead of 'catalysis').
4. Avoid generic or vague terms (e.g., 'study', 'analysis', 'process').
5. Be formatted as a JSON list of exactly 5–10 unique entries without explanation.
"""

FEW_SHOT = """
### Example
Input:
Title/Topic: CO2 Electroreduction to Multicarbon Products on Copper Nanocubes
Authors/Affiliations: M. Garcia (ETH Zürich, Switzerland)
Abstract: Copper nanocube electrodes selectively reduce CO2 to C2+ products via...
Output JSON:
{"keywords": ["CO2 electroreduction", "copper nanocubes", "C2+ products", "electrocatalysis", "selectivity tuning"],
 "countries": ["Switzerland"]}
"""

User_Prompt_template = """
{few_shot}
Extract exactly 5 to 10 most important chemical engineering keywords and all author countries from this abstract.
Format as JSON:
{{"keywords": ["keyword1", "keyword2", "keyword3", "keyword4", "keyword5"],
  "countries": ["country1", "country2", ...]}}

Input:
Title/Topic: {title_or_topic}
Authors/Affiliations: {authors_block}
Abstract: {abstract_text}

Output JSON:
"""

# ---------------------------------------------------------
# 3. Logic & API Call Handling
# ---------------------------------------------------------


def _call_openrouter_direct(messages: list) -> str:
    """Uses requests to call OpenRouter. Handles 429 and HTML errors."""
    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json",
        "HTTP-Referer": "http://localhost:3000",
        "X-Title": "AIChE Keyword Extractor"
    }
    payload = {
        "model": MODEL_ID,
        "messages": messages,
        "temperature": 0.0
    }
    
    delay = 2
    for attempt in range(5):
        try:
            response = requests.post(OPENROUTER_URL, headers=headers, json=payload, timeout=60)
            
            if response.status_code == 429:
                print(f"\n[429 Rate Limit] Waiting {delay}s...")
                time.sleep(delay)
                delay *= 2
                continue
                
            if "text/html" in response.headers.get("Content-Type", ""):
                print(f"\n[API Error] Received HTML. Status: {response.status_code}")
                return ""
                
            response.raise_for_status()
            res_json = response.json()
            return res_json['choices'][0]['message']['content']
            
        except Exception as e:
            print(f"\n[Attempt {attempt+1}] Error: {e}")
            time.sleep(2)
    return ""

def _extract_json_block(text: str) -> Dict[str, Any]:
    """Robustly extracts JSON even if surrounded by prose."""
    match = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if not match: return {}
    try:
        return json.loads(match.group(0).strip("`"))
    except:
        return {}

def _format_authors(item: Dict[str, Any]) -> str:
    """Formats author data for the prompt."""
    authors = item.get("authors_structured", [])
    if isinstance(authors, list) and len(authors) > 0:
        return "; ".join([f"{a.get('name')} ({a.get('affiliation')})" for a in authors if a])[:800]
    return str(item.get("presenting_author", "Unknown Author"))[:800]

# ---------------------------------------------------------
# 4. Main Batch Processing
# ---------------------------------------------------------
def process_file(file_path: str):
    fname = os.path.basename(file_path)
    print(f"\n{'='*60}\nFILE: {fname}\n{'='*60}")
    
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    
    results = []
    total = len(data)

    for i, item in enumerate(data, 1):
        title = (item.get("topic") or item.get("title") or "No Title").strip()
        abstract = (item.get("abstract") or "").strip()
        authors_txt = _format_authors(item)
        
        prompt = User_Prompt_template.format(
            few_shot=FEW_SHOT.strip(), 
            title_or_topic=title,
            authors_block=authors_txt, 
            abstract_text=abstract
        )
        
        messages = [
            {"role": "system", "content": System_Prompt.strip()},
            {"role": "user", "content": prompt}
        ]
        
        raw_output = _call_openrouter_direct(messages)
        parsed = _extract_json_block(raw_output)
        
        results.append({
            "title": title,
            "authors": authors_txt,
            "keywords": parsed.get("keywords", []),
            "countries": parsed.get("countries", []),
            "Year": ABSTRACT_YEAR
        })
        
        # Notebook progress indicator
        if i % 5 == 0 or i == total:
            print(f"  > Progress: {i}/{total} items processed...", end="\r")
        
        # Polite delay to keep the free API session healthy
        time.sleep(0.6)
        
    print(f"\n\nFinished {fname}.")
    return results

if __name__ == "__main__":
    os.makedirs(OUTPUT_FOLDER, exist_ok=True)
    
    if not os.path.exists(INPUT_FOLDER):
        print(f"Error: Folder '{INPUT_FOLDER}' not found.")
    else:
        json_files = sorted([f for f in os.listdir(INPUT_FOLDER) if f.endswith(".json")])
        master_list = []
        
        for filename in json_files:
            file_results = process_file(os.path.join(INPUT_FOLDER, filename))
            master_list.extend(file_results)
            
            # Save individual file result
            out_path = os.path.join(OUTPUT_FOLDER, f"{os.path.splitext(filename)[0]}_extracted.json")
            with open(out_path, "w", encoding="utf-8") as f:
                json.dump(file_results, f, ensure_ascii=False, indent=2)

        # Final Master Export
        master_path = os.path.join(OUTPUT_FOLDER, "all_combined_extracted.json")
        with open(master_path, "w", encoding="utf-8") as f:
            json.dump(master_list, f, ensure_ascii=False, indent=2)
            
        print(f"\n{'#'*60}\nBATCH COMPLETE. Master File: {master_path}\n{'#'*60}")


FILE: aiche_papers_3288.json

[429 Rate Limit] Waiting 2s...

[429 Rate Limit] Waiting 4s...

[429 Rate Limit] Waiting 8s...

[429 Rate Limit] Waiting 16s...

[429 Rate Limit] Waiting 32s...


KeyboardInterrupt: 

In [ ]:
import os
import json
import re
import time
import requests
from typing import List, Dict, Any
from dotenv import load_dotenv

# ---------------------------------------------------------
# 1. Configuration & Client Setup
# ---------------------------------------------------------
load_dotenv()
API_KEY = os.getenv("OPENROUTER_API_KEY")
if not API_KEY:
    raise RuntimeError("OPENROUTER_API_KEY not found. Check your .env file.")

OPENROUTER_URL = "https://openrouter.ai/v1/chat/completions"

# THE REQUESTED MODELS (Trimmed to 3 fallbacks to satisfy OpenRouter API constraints)
PRIMARY_MODEL = "meta-llama/llama-3.3-70b-instruct:free"
FALLBACK_CHAIN = [
    "meta-llama/llama-3.1-405b-instruct:free",
    "qwen/qwen3-coder:free",
    "deepseek/deepseek-r1-0528:free"
]

ABSTRACT_YEAR = 2023
INPUT_FOLDER = "aiche_file"
OUTPUT_FOLDER = "openrouter_extracted"

# ---------------------------------------------------------
# 2. Prompts (STRICTLY UNCHANGED)
# ---------------------------------------------------------
System_Prompt = """
You are an expert keyword extractor specialized in chemical engineering.
Your task is to analyze abstracts from AIChE conferences and extract
exactly 5 or 10 of the most important and specific keywords or phrases related to
chemical engineering from each abstract, along with the unique countries
of all authors.

The keywords should:
1. Reflect the core chemical engineering focus of the abstract.
2. Each keyword should be either one word or two words—no longer phrases allowed.
3. Be highly specific (e.g., 'heterogeneous catalysis' instead of 'catalysis').
4. Avoid generic or vague terms (e.g., 'study', 'analysis', 'process').
5. Be formatted as a JSON list of exactly 5–10 unique entries without explanation.
"""

FEW_SHOT = """
### Example
Input:
Title/Topic: CO2 Electroreduction to Multicarbon Products on Copper Nanocubes
Authors/Affiliations: M. Garcia (ETH Zürich, Switzerland)
Abstract: Copper nanocube electrodes selectively reduce CO2 to C2+ products via...
Output JSON:
{"keywords": ["CO2 electroreduction", "copper nanocubes", "C2+ products", "electrocatalysis", "selectivity tuning"],
 "countries": ["Switzerland"]}
"""

User_Prompt_template = """
{few_shot}
Extract exactly 5 to 10 most important chemical engineering keywords and all author countries from this abstract.
Format as JSON:
{{"keywords": ["keyword1", "keyword2", "keyword3", "keyword4", "keyword5"],
  "countries": ["country1", "country2", ...]}}

Input:
Title/Topic: {title_or_topic}
Authors/Affiliations: {authors_block}
Abstract: {abstract_text}

Output JSON:
"""

# ---------------------------------------------------------
# 3. Robust API & Parsing Logic
# ---------------------------------------------------------

def _call_openrouter_api(messages: list) -> str:
    """Uses direct requests with server-side fallbacks and logs model transitions."""
    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json",
        "HTTP-Referer": "http://localhost:3000",
        "X-Title": "AIChE Keyword Agent"
    }
    
    payload = {
        "model": PRIMARY_MODEL,
        "models": FALLBACK_CHAIN, 
        "messages": messages,
        "temperature": 0.0
    }
    
    delay = 5
    for attempt in range(5):
        try:
            response = requests.post(OPENROUTER_URL, headers=headers, json=payload, timeout=120)
            
            if response.status_code == 429:
                print(f"\n[429 Rate Limit] Model {PRIMARY_MODEL} busy. Sleeping {delay}s...")
                time.sleep(delay)
                delay *= 2
                continue
            
            if response.status_code == 402:
                print(f"\n[402 Error] Account verification required for {PRIMARY_MODEL}.")
                return ""
            
            if response.status_code == 404:
                print(f"\n[404 Error] Model ID not found. Current: {PRIMARY_MODEL}. Retrying...")
                time.sleep(1)
                continue

            response.raise_for_status()
            res_json = response.json()
            
            # TRACKING FALLBACKS: Check which model actually provided the result
            final_model = res_json.get("model", "unknown")
            if final_model != PRIMARY_MODEL:
                print(f" (Switching: Used {final_model})", end="")
            else:
                print(f" (Using: {final_model})", end="")
            
            return res_json['choices'][0]['message']['content']
            
        except Exception as e:
            # PRINTING MODEL ON ERROR: Identify where the failure occurred
            print(f"\n[Attempt {attempt+1}] Connection Error with {PRIMARY_MODEL}: {e}")
            time.sleep(2)
    return ""

def _extract_final_json(text: str) -> Dict[str, Any]:
    try:
        matches = re.findall(r"\{.*\}", text, flags=re.DOTALL)
        if matches:
            return json.loads(matches[-1].strip("`"))
    except:
        pass
    return {}

def _format_authors_block(item: Dict[str, Any]) -> str:
    authors = item.get("authors_structured", [])
    if isinstance(authors, list) and len(authors) > 0:
        return "; ".join([f"{a.get('name')} ({a.get('affiliation')})" for a in authors if a])[:800]
    return str(item.get("presenting_author", "Unknown"))[:800]

# ---------------------------------------------------------
# 4. Processing Functions
# ---------------------------------------------------------

def process_single_file(file_path: str):
    fname = os.path.basename(file_path)
    print(f"\n{'='*60}\nFILE: {fname}\n{'='*60}")
    
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    
    results = []
    total = len(data)

    for i, item in enumerate(data, 1):
        title = (item.get("topic") or item.get("title") or "No Title").strip()
        abstract = (item.get("abstract") or "").strip()
        authors = _format_authors_block(item)
        
        prompt = User_Prompt_template.format(
            few_shot=FEW_SHOT.strip(), title_or_topic=title,
            authors_block=authors, abstract_text=abstract
        )
        
        # UI Log to see the current progress and attempt
        print(f"  > Progress: {i}/{total} Handling: {title[:40]}...", end="")
        
        raw_output = _call_openrouter_api([
            {"role":"system","content":System_Prompt.strip()},
            {"role":"user","content":prompt}
        ])
        
        parsed = _extract_final_json(raw_output)
        
        results.append({
            "title": title,
            "authors": authors,
            "keywords": parsed.get("keywords", []),
            "countries": parsed.get("countries", []),
            "Year": ABSTRACT_YEAR
        })
        
        print("\r", end="") # Clear the line for the next item
        time.sleep(10) 
        
    print(f"\nFinished {fname}.")
    return results

if __name__ == "__main__":
    os.makedirs(OUTPUT_FOLDER, exist_ok=True)
    
    if not os.path.exists(INPUT_FOLDER):
        print(f"ERROR: Input directory '{INPUT_FOLDER}' not found.")
    else:
        json_files = sorted([f for f in os.listdir(INPUT_FOLDER) if f.endswith(".json")])
        master_combined = []
        
        print(f"STARTING BATCH: {len(json_files)} files found.")

        for filename in json_files:
            file_results = process_single_file(os.path.join(INPUT_FOLDER, filename))
            master_combined.extend(file_results)

            with open(os.path.join(OUTPUT_FOLDER, f"{os.path.splitext(filename)[0]}_results.json"), "w", encoding="utf-8") as f:
                json.dump(file_results, f, indent=2)

        with open(os.path.join(OUTPUT_FOLDER, "all_combined_extracted.json"), "w", encoding="utf-8") as f:
            json.dump(master_combined, f, indent=2)
            
        print(f"\nBATCH COMPLETE. Master File: {OUTPUT_FOLDER}/all_combined_extracted.json")

STARTING BATCH: 22 files found.

FILE: aiche_papers_3288.json
  > Progress: 1/65 Handling: 224a- Predicting the Vapor-Liquid Equili...
[Attempt 1] Connection Error with meta-llama/llama-3.3-70b-instruct:free: Expecting value: line 1 column 1 (char 0)

[Attempt 2] Connection Error with meta-llama/llama-3.3-70b-instruct:free: Expecting value: line 1 column 1 (char 0)

[Attempt 3] Connection Error with meta-llama/llama-3.3-70b-instruct:free: Expecting value: line 1 column 1 (char 0)


KeyboardInterrupt: 